# Team VizLaLune: Milano-Cortina 2026 Olympic Winter Games


| Student's name | SCIPER |
| -------------- | ------ |
| Thomas Lenges  | 325245 |
| Léontine Lefranc | |
| Manon Darnaud | |

This Jupyter Notebook serves to pre-process the data from the Kaggle dataset *["Milano-Cortina 2026 Olympic Winter Games"](https://www.kaggle.com/datasets/piterfm/milano-cortina-2026-olympic-winter-games/data?select=medals.csv)*.

This pre-processing bases itself on (see *[EXPLORATION_Milano_Cortina_2026_Olympic_Winter_Games.ipynb](https://github.com/com-480-data-visualization/VizLaLune)*)

The pre-processing leads to final `.csv` files to be used for visualization (see *[XXX](https://github.com/com-480-data-visualization/VizLaLune)*)

The dataset has the following files:

- athletes.csv
- coaches.csv
- medallists.csv
- medals.csv
- schedules.csv
- teams.csv
- venues.csv

# Download dataset

In [70]:
# Download dataset from Kaggle using kagglehub library
import kagglehub

path = kagglehub.dataset_download("piterfm/milano-cortina-2026-olympic-winter-games")

print("Path to dataset files:", path)

Path to dataset files: /home/thomas/.cache/kagglehub/datasets/piterfm/milano-cortina-2026-olympic-winter-games/versions/11


# Pre-processing

In [71]:
# To avoid re-download the dataset (need to be changed by user)
path = "/home/thomas/.cache/kagglehub/datasets/piterfm/milano-cortina-2026-olympic-winter-games/versions/11"

In [72]:
# Load the dataset using pandas
import pandas as pd

df_athletes = pd.read_csv(path + "/athletes.csv")
df_coaches = pd.read_csv(path + "/coaches.csv")
df_medallists = pd.read_csv(path + "/medallists.csv")
df_medals = pd.read_csv(path + "/medals.csv")
df_schedules = pd.read_csv(path + "/schedules.csv")
df_teams = pd.read_csv(path + "/teams.csv")
df_venues = pd.read_csv(path + "/venues.csv")

## Athletes

### From the data exploration this sub-dataset requires the following pre-processing:

* Deal with the missing event for some athletes

In [73]:
import ast

def has_empty_event(events_str):
    events_list = ast.literal_eval(events_str)
    return any(event["event"] == "" for event in events_list)

# Filter empty event athletes
df_athletes_missing_events = df_athletes[df_athletes["events"].apply(has_empty_event)]

print("Athletes with missing events: " + str(len(df_athletes_missing_events)))
df_athletes_missing_events

Athletes with missing events: 63


,code,name,gender,country_code,flag_bearer,events
5,23307,ABDUL-SABOOR Hakeem,M,USA,0,"[{'discipline': 'Bobsleigh', 'event': ''}]"
95,53404,ASSELIN Olivia,F,CAN,0,"[{'discipline': 'Freestyle Skiing', 'event': ''}]"
156,47746,BEBRISS Arnis,M,LAT,0,"[{'discipline': 'Bobsleigh', 'event': ''}]"
158,20034,BECKER Tim,M,GER,0,"[{'discipline': 'Bobsleigh', 'event': ''}]"
164,54946,BEKTAS Ozan,M,LIE,0,"[{'discipline': 'Bobsleigh', 'event': ''}]"
...,...,...,...,...,...,...
2737,147641,WANG Xinyi,F,CHN,0,"[{'discipline': 'Bobsleigh', 'event': ''}]"
2752,41500,WEIDEL Anna,F,GER,0,"[{'discipline': 'Biathlon', 'event': ''}]"
2795,52954,WILSON Eden,F,CAN,0,"[{'discipline': 'Bobsleigh', 'event': ''}]"
2851,23903,YOUNG Jack,M,USA,0,"[{'discipline': 'Cross-Country Skiing', 'event..."


In [74]:
df_athletes_missing_events.groupby("events").size()

events
[{'discipline': 'Alpine Skiing', 'event': ''}]            8
[{'discipline': 'Biathlon', 'event': ''}]                13
[{'discipline': 'Bobsleigh', 'event': ''}]               39
[{'discipline': 'Cross-Country Skiing', 'event': ''}]     2
[{'discipline': 'Freestyle Skiing', 'event': ''}]         1
dtype: int64

In [75]:
df_athletes_missing_events.groupby("country_code").size()

country_code
AND    1
AUS    1
AUT    3
BRA    1
CAN    4
CHN    2
FIN    2
FRA    5
GBR    2
GER    6
ISR    1
ITA    3
JAM    1
KOR    2
LAT    5
LIE    1
NED    1
NOR    4
POL    1
SLO    1
SUI    6
SWE    3
TPE    1
TTO    1
USA    5
dtype: int64

After some investigation. The reasons why these athletes have a missing event are various: alternate athlete (compete if another athlete withdraws or gets injured)(e.g. ABDUL-SABOOR Hakeem)/injury (e.g. ASSELIN Olivia) etc.

Cannot be filled as no information can be found regarding the exact event, only the discipline can be found and is already present.

Hence one can decide to keep such athletes for the moment as some still came with their delegation to the games.

### Things to keep in mind for visualizations:

-Some athletes did not actually participate

-Some athletes have multiple events

In [76]:
df_athletes.head()

,code,name,gender,country_code,flag_bearer,events
0,139893,AABREKK Ingrid Bergene,F,NOR,0,"[{'discipline': 'Cross-Country Skiing', 'event..."
1,24464,AAGAARD Mikkel,M,DEN,0,"[{'discipline': 'Ice Hockey', 'event': 'Men'}]"
2,45083,AALTO Antti,M,FIN,0,"[{'discipline': 'Ski Jumping', 'event': ""Men's..."
3,32319,ABATANGELO Aurora,F,ITA,0,"[{'discipline': 'Ice Hockey', 'event': 'Women'}]"
4,4972,ABDI Fayik,M,KSA,0,"[{'discipline': 'Alpine Skiing', 'event': ""Men..."


In [77]:
# Change path according to user
path = "/home/thomas/Documents/Spring26/COM-480/Project/DataPreprocessing"

df_athletes.to_csv(path + "/athletes.csv", index=False)

## Coaches

### From the data exploration this sub-dataset requires the following pre-processing:

* None (this dataset is simply incomplete and hard to fill (all data are directly fetched from the [official Milano-Cortina 2026 Winter Olympic Games website](https://www.olympics.com/en/milano-cortina-2026/results/hubs/individuals/coaches))).

In [78]:
# Extract disciplines from coaches events
disciplines = []

for events_str in df_coaches["events"]:
    events_list = ast.literal_eval(events_str)
    for event in events_list:
        disciplines.append(event["discipline"])

# Count by discipline
discipline_counts = pd.Series(disciplines).value_counts()
print(discipline_counts)

Curling       33
Ice Hockey    22
Name: count, dtype: int64


### Things to keep in mind for visualizations:

-Dataset is incomplete

-Dataset focuses solely on Curling and Ice Hockey

In [79]:
df_coaches.head()

,code,name,gender,country_code,nationality,function,events
0,50140,BOUCHARD Eric,M,ITA,CAN,Head Coach,"[{'discipline': 'Ice Hockey', 'event': 'Women'}]"
1,50988,CERNOVSKY Vladimir,M,CZE,CZE,Coach,"[{'discipline': 'Curling', 'event': 'Mixed Dou..."
2,49020,CHARETTE Pierre,M,SUI,CAN,Coach,"[{'discipline': 'Curling', 'event': 'Women'}]"
3,53809,COOPER Jon,M,CAN,CAN,Head Coach,"[{'discipline': 'Ice Hockey', 'event': 'Men'}]"
4,49035,de CRUZ Peter,M,SUI,SUI,Coach,"[{'discipline': 'Curling', 'event': 'Mixed Dou..."


In [80]:
# Change path according to user

df_coaches.to_csv(path + "/coaches.csv", index=False)

CHANGE LINKS (TWO NOTEBOOKS + MILESTONE 1)

## Medallists

### From the data exploration this sub-dataset requires the following pre-processing:

* None (sub-dataset is complete!). As for all sections in this notebook, check the exploration.

### Things to keep in mind for visualizations:

-One line per medal received (multiple rows for same athlete if multiple medals received)

-More medallists than events due to team events

In [81]:
df_medallists.head()

,date,medal_code,medal,code,name,gender,country_code,country,discipline,discipline_code,event_name
0,2026-02-07,1,GOLD,48729,von ALLMEN Franjo,M,SUI,Switzerland,Alpine Skiing,ALP,Men's Downhill
1,2026-02-07,2,SILVER,25998,FRANZONI Giovanni,M,ITA,Italy,Alpine Skiing,ALP,Men's Downhill
2,2026-02-07,3,BRONZE,26002,PARIS Dominik,M,ITA,Italy,Alpine Skiing,ALP,Men's Downhill
3,2026-02-07,1,GOLD,30739,KARLSSON Frida,F,SWE,Sweden,Cross-Country Skiing,CCS,Women's 10km + 10km Skiathlon
4,2026-02-07,2,SILVER,30707,ANDERSSON Ebba,F,SWE,Sweden,Cross-Country Skiing,CCS,Women's 10km + 10km Skiathlon


In [82]:
df_medallists.to_csv(path + "/medallists.csv", index=False)

## Medals

### From the data exploration this sub-dataset requires the following pre-processing:

* None (sub-dataset is complete!). As for all sections in this notebook, check the exploration.

### Things to keep in mind for visualizations:

-Similar ranks due to ties. Can also exploit rank_total (medal sum instead of first gold medal sum then silver then bronze) instead of rank. 

-Only one medal per sport not per individual (709 vs. 348)

In [83]:
df_medals.head()

,country,country_code,gold,silver,bronze,total,rank,rank_total
0,Norway,NOR,18,12,11,41,1,1
1,United States,USA,12,12,9,33,2,2
2,Netherlands,NED,10,7,3,20,3,9
3,Italy,ITA,10,6,14,30,4,3
4,Germany,GER,8,10,8,26,5,4


In [84]:
df_medals.to_csv(path + "/medals.csv", index=False)

## Schedules

### From the data exploration this sub-dataset requires the following pre-processing:

* Get rid of duplicated rows

* Keep only FINISHED "status"

* Get rid of ceremonies for "event"

* Rename appropriately some events

* Drop the rows with 3 for "event_medal" (which are essentially duplicates)

See data exploration for all explainations

In [85]:
# Get rid of duplicated rows
df_schedules = df_schedules.drop_duplicates()

In [86]:
# Keep only FINISHED "status"
df_schedules = df_schedules[df_schedules["status"] == "FINISHED"]

In [87]:
# Get rid of ceremonies for "event"
df_schedules = df_schedules[~df_schedules["event"].str.contains("ceremon", case=False)]

In [88]:
# Rename appropriately some events
df_schedules["event"] = df_schedules["event"].replace("Large Hill Training", "Individual Gundersen Large Hill/10km")
df_schedules["event"] = df_schedules["event"].replace("Normal Hill Training", "Individual Gundersen Normal Hill/10km")
df_schedules["event"] = df_schedules["event"].replace("Men's Large Hill", "Men's LH Individual")
df_schedules["event"] = df_schedules["event"].replace("Men's Normal Hill", "Men's NH Individual")
df_schedules["event"] = df_schedules["event"].replace("Women's Large Hill", "Women's LH Individual")
df_schedules["event"] = df_schedules["event"].replace("Women's Normal Hill", "Women's NH Individual")

In [89]:
# Drop the rows with 3 for "event_medal"
df_schedules = df_schedules[df_schedules["event_medal"] != 3]

### Things to keep in mind for visualizations:

-Some events may not have all their trainings hence using only medal events may be safer

In [90]:
df_schedules.head()

,start_date,end_date,day,status,discipline,discipline_code,event,event_medal,phase,gender,event_type,venue,venue_code,location,location_code,id
0,2026-02-04T11:30:00+01:00,2026-02-04T13:30:00+01:00,2026-02-04,FINISHED,Alpine Skiing,ALP,Men's Downhill,0,Men's Downhill Official Training,M,INDV,Stelvio Ski Centre,SSC,Stelvio Ski Centre-Alpine Skiing Course,SAL,ALPMDH----------------TRNO000100--
1,2026-02-04T19:05:00+01:00,2026-02-04T21:00:00+01:00,2026-02-04,FINISHED,Curling,CUR,Mixed Doubles,0,Mixed Doubles Round Robin,X,TEAM,Cortina Curling Olympic Stadium,CCU,Cortina Curling Olympic Stadium- Sheet A,CUA,CURXTEAM2-------------PREL000101--
2,2026-02-04T19:05:00+01:00,2026-02-04T21:00:00+01:00,2026-02-04,FINISHED,Curling,CUR,Mixed Doubles,0,Mixed Doubles Round Robin,X,TEAM,Cortina Curling Olympic Stadium,CCU,Cortina Curling Olympic Stadium- Sheet B,CUB,CURXTEAM2-------------PREL000102--
3,2026-02-04T19:05:00+01:00,2026-02-04T21:00:00+01:00,2026-02-04,FINISHED,Curling,CUR,Mixed Doubles,0,Mixed Doubles Round Robin,X,TEAM,Cortina Curling Olympic Stadium,CCU,Cortina Curling Olympic Stadium- Sheet C,CUC,CURXTEAM2-------------PREL000103--
4,2026-02-04T19:05:00+01:00,2026-02-04T21:00:00+01:00,2026-02-04,FINISHED,Curling,CUR,Mixed Doubles,0,Mixed Doubles Round Robin,X,TEAM,Cortina Curling Olympic Stadium,CCU,Cortina Curling Olympic Stadium- Sheet D,CUD,CURXTEAM2-------------PREL000104--


In [91]:
df_schedules.to_csv(path + "/schedules.csv", index=False)

## Teams

### From the data exploration this sub-dataset requires the following pre-processing:

* Complete gender data (keep "X" incase it is a mixed event and is alread "-" or "X")

In [92]:
df_teams.groupby("gender").size()

gender
M    217
W    165
X    168
dtype: int64

In [93]:
def has_man_event(events_str):
    """Check if events contain 'man'/'men' but NOT 'woman'/'women'"""
    events_list = ast.literal_eval(events_str)
    keywords = ["man", "men"]
    
    for event in events_list:
        event_name = event["event"].lower()
        # Check if keyword exists AND neither 'woman' nor 'women' exist
        if any(keyword in event_name for keyword in keywords) and "woman" not in event_name and "women" not in event_name:
            return True
    return False

# Update gender to "M" for men events with missing gender (-)
mask = (df_teams["gender"] == "-") & (df_teams["events"].apply(has_man_event))
df_teams.loc[mask, "gender"] = "M"

print(f"Updated {mask.sum()} teams with 'man/men' events to gender 'M'")

Updated 0 teams with 'man/men' events to gender 'M'


In [94]:
def has_woman_event(events_str):
    """Check if events contain 'woman'/'women' but NOT 'man'/'men'"""
    events_list = ast.literal_eval(events_str)
    keywords = ["woman", "women"]
    
    for event in events_list:
        event_name = event["event"].lower()
        # Don't need to check for men/man anymore as - with man/men have already been updated
        if any(keyword in event_name for keyword in keywords):
            return True
    return False

# Update gender to "W" for woman events with missing gender (-)
mask = (df_teams["gender"] == "-") & (df_teams["events"].apply(has_woman_event))
df_teams.loc[mask, "gender"] = "W"

print(f"Updated {mask.sum()} teams with 'woman/women' events to gender 'W'")

Updated 0 teams with 'woman/women' events to gender 'W'


In [95]:
def has_pair_event(events_str):
    """Check if events contain 'pair'"""
    events_list = ast.literal_eval(events_str)
    keywords = ["pair", "mixed"]
    return any(keyword in event["event"].lower() for keyword in keywords for event in events_list)

# Update gender to "X" for mixed events (pair) with missing gender (-)
mask = (df_teams["gender"] == "-") & (df_teams["events"].apply(has_pair_event))

df_teams.loc[mask, "gender"] = "X"

print(f"Updated {mask.sum()} teams with 'pair/mixed' events to gender 'X'")

Updated 0 teams with 'pair/mixed' events to gender 'X'


In [96]:
no_gender = df_teams[df_teams["gender"] == "-"]
no_gender.groupby("events").size()

Series([], dtype: int64)

In [97]:
# For simplicity, we can update the remaining genders to "X"

mask = df_teams["gender"] == "-"

df_teams.loc[mask, "gender"] = "X"

print(f"Updated {mask.sum()} remaining teams to gender 'X'")

Updated 0 remaining teams to gender 'X'


In [98]:
df_teams.groupby("gender").size()

gender
M    217
W    165
X    168
dtype: int64

### Things to keep in mind for visualizations:

-Still have a lot of unknown genders

-Incomplete dataset. Missing a lot of teams (missing countries as well as events)

-Poor potential use

In [99]:
df_teams.head()

,code,name,team_type,gender,country_code,events
0,FSKXPAIRS---ARM01,AKOPOVA Karina / RAKHMANIN Nikita,CPLW,X,ARM,"[{'discipline': 'Figure Skating', 'event': 'Pa..."
1,BOBMTEAM2---GER03,AMMOUR Adam,CUSTOM,M,GER,"[{'discipline': 'Bobsleigh', 'event': '2-man'}]"
2,BOBMTEAM4---GER03,AMMOUR Adam,CUSTOM,M,GER,"[{'discipline': 'Bobsleigh', 'event': '4-man'}]"
3,BOBWTEAM2---ITA01,ANDREUTTI Giada,CUSTOM,W,ITA,"[{'discipline': 'Bobsleigh', 'event': '2-woman'}]"
4,SKNXRELAY2--LAT01,ANDZANE Marta / INDRIKSONS Emils,CPLW,X,LAT,"[{'discipline': 'Skeleton', 'event': 'Mixed Te..."


In [100]:
df_teams.to_csv(path + "/teams.csv", index=False)

## Venues

### From the data exploration this sub-dataset requires the following pre-processing:

* Update missing url_sport

* Add event column for precise event name (use schedules.csv for this)

Cannot simply add all events linked to a discipline as some disciplines have their events split among different venues

In [101]:
# Update missing url_sport
def get_discipline_urls(discipline_str):
    if pd.isna(discipline_str):
        return None
    
    try:
        disciplines_list = ast.literal_eval(discipline_str)
    except:
        disciplines_list = [discipline_str]
    
    return ["https://www.olympics.com/en/milano-cortina-2026/sports/" + d.partition("-")[2] + "/" for d in disciplines_list] # Want to keep only sport not with discipline-sport (watch out because some sports also have "-" within their name)

df_venues["url_sport"] = df_venues["disciplines"].apply(get_discipline_urls)

In [102]:
# Create a dictionary: venue_code -> list of (discipline, event) pairs
venue_events = {}

for _, row in df_schedules.iterrows():
    venue_code = row["venue_code"]
    discipline = row["discipline"]
    event = row["event"]
    
    # If venue not yet in dict, create empty list otherwise error
    if venue_code not in venue_events:
        venue_events[venue_code] = []
    
    # Add discipline:event pair (avoid duplicates)
    pair = (discipline, event)
    if pair not in venue_events[venue_code]:
        venue_events[venue_code].append(pair)

# Now add this to df_venues as a new column
df_venues["events"] = df_venues["venue_code"].apply(lambda code: venue_events.get(code, []))

### Things to keep in mind for visualizations:

-Complete dataset

In [103]:
df_venues.head()

,venue,venue_code,disciplines,disciplines_code,url_sport,url_venue,events
0,Anterselva Biathlon Arena,ABA,discipline-biathlon,BTH,[https://www.olympics.com/en/milano-cortina-20...,https://www.olympics.com/en/milano-cortina-202...,"[(Biathlon, Mixed Relay 4 x 6km (M+W)), (Biath..."
1,Cortina Curling Olympic Stadium,CCU,discipline-curling,CUR,[https://www.olympics.com/en/milano-cortina-20...,https://www.olympics.com/en/milano-cortina-202...,"[(Curling, Mixed Doubles), (Curling, Men), (Cu..."
2,Cortina Sliding Centre,CSC,"[""discipline-bobsleigh"",""discipline-luge"",""dis...","[""BOB"",""LUG"",""SKN""]",[https://www.olympics.com/en/milano-cortina-20...,https://www.olympics.com/en/milano-cortina-202...,"[(Luge, Men's Singles), (Luge, Women's Singles..."
3,Livigno Aerials & Moguls Park,LAM,discipline-freestyle-skiing,FRS,[https://www.olympics.com/en/milano-cortina-20...,https://www.olympics.com/en/milano-cortina-202...,"[(Freestyle Skiing, Men's Moguls), (Freestyle ..."
4,Livigno Snow Park,LSP,"[""discipline-freestyle-skiing"",""discipline-sno...","[""FRS"",""SBD""]",[https://www.olympics.com/en/milano-cortina-20...,https://www.olympics.com/en/milano-cortina-202...,"[(Snowboard, Men's Snowboard Big Air), (Freest..."


In [104]:
df_venues.to_csv(path + "/venues.csv", index=False)